# Stage B — Breast Histopathology Image Classifier (BreakHis)

**Goal:** prove our deep-learning image pipeline works by teaching a neural network to tell
**benign vs. malignant** breast-tumor microscope images apart — on clean, pre-cropped data,
before we tackle the much harder gigapixel TCGA slides in Stage C.

**How to run this:**
1. Open this file in [Google Colab](https://colab.research.google.com) (File > Upload notebook).
2. Turn on the free GPU: **Runtime > Change runtime type > Hardware accelerator = GPU**.
3. Run the cells top to bottom (Runtime > Run all).

**Honesty note:** BreakHis is a *classification* warm-up (benign vs malignant). It validates
that we can extract signal from histology images. It is **not** a survival dataset — the
survival question comes in Stage C on TCGA slides.

### Step 0 — Check we actually have a GPU
If this prints an empty list, go enable the GPU (see above) or training will be very slow.

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

### Step 1 — Download the BreakHis dataset (~4 GB)
Public, non-commercial research use. Cite Spanhol et al., IEEE TBME 2016.

In [ ]:
import os, urllib.request, tarfile
URL = 'http://www.inf.ufpr.br/vri/databases/BreaKHis_v1.tar.gz'
if not os.path.exists('BreaKHis_v1'):
    print('Downloading (a few minutes)...')
    urllib.request.urlretrieve(URL, 'BreaKHis_v1.tar.gz')
    print('Extracting...')
    with tarfile.open('BreaKHis_v1.tar.gz') as t:
        t.extractall('.')
    print('Done.')
else:
    print('Already downloaded.')

### Step 2 — Catalog the images (label + PATIENT id)

This is the most important rigor step. Each filename looks like `SOB_B_TA-14-21978AB-40-001.png`:
- `B`/`M` and the folder tell us **benign** vs **malignant**,
- `14-21978AB` is the **patient/slide id**,
- `40` is the magnification (40x/100x/200x/400x).

We record the patient id so we can keep **all of one patient's images on the same side** of the
train/test split. If we didn't, the model could 'memorize' a patient seen in training and we'd
get a fake-high score (data leakage).

In [ ]:
import glob, pandas as pd
MAG = '200'   # use 200x images (change to 40/100/400 to compare)
rows = []
for path in glob.glob('BreaKHis_v1/**/*.png', recursive=True):
    fname = os.path.basename(path)
    parts = fname.split('-')
    if len(parts) < 5:
        continue
    mag = parts[3]
    if mag != MAG:
        continue
    label = 1 if '/malignant/' in path.lower() else 0   # 1=malignant, 0=benign
    patient = parts[1] + '-' + parts[2]
    rows.append({'path': path, 'label': label, 'patient': patient})
df = pd.DataFrame(rows)
print('images:', len(df), '| patients:', df.patient.nunique())
print(df.label.value_counts().rename({0:'benign',1:'malignant'}))

### Step 3 — Patient-level train / validation / test split
No patient appears in more than one split (prevents leakage).

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
SEED = 42
# first split off 20% of PATIENTS as test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
trainval_idx, test_idx = next(gss.split(df, groups=df.patient))
trainval, test = df.iloc[trainval_idx], df.iloc[test_idx]
# then 20% of the remaining patients as validation
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, val_idx = next(gss2.split(trainval, groups=trainval.patient))
train, val = trainval.iloc[tr_idx], trainval.iloc[val_idx]
for name, part in [('train',train),('val',val),('test',test)]:
    print(f'{name}: {len(part)} images, {part.patient.nunique()} patients')
# sanity: no patient overlap across splits
assert not (set(train.patient) & set(test.patient))
assert not (set(train.patient) & set(val.patient))
print('No patient overlap across splits.')

### Step 4 — Build fast image pipelines (tf.data)
Decode PNG, resize to 224x224, light augmentation on the training set only.

In [ ]:
IMG = 224; BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE
def load(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG, IMG])
    return img, label   # EfficientNet expects 0-255 inputs (it normalizes internally)
def make_ds(part, training=False):
    ds = tf.data.Dataset.from_tensor_slices((part.path.values, part.label.values.astype('float32')))
    if training:
        ds = ds.shuffle(1000, seed=SEED)
    ds = ds.map(load, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
    return ds
train_ds, val_ds, test_ds = make_ds(train, True), make_ds(val), make_ds(test)

### Step 5 — The model (transfer learning)

We start from **EfficientNetB0**, a network already trained on millions of everyday images.
It already knows edges, textures, and shapes; we freeze that knowledge and train just a small
new 'head' to map those features to benign/malignant. This is why it works with limited data.

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
base = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG,IMG,3), pooling='avg')
base.trainable = False   # freeze the pretrained features for now
inp = layers.Input((IMG,IMG,3))
x = layers.RandomFlip('horizontal_and_vertical')(inp)
x = layers.RandomRotation(0.1)(x)
x = base(x, training=False)
x = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation='sigmoid')(x)
model = Model(inp, out)
model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
model.summary()

### Step 6 — Train
We weight the classes because there are more malignant than benign images, so the model can't win by always guessing 'malignant'.

In [ ]:
import numpy as np
n0 = (train.label==0).sum(); n1 = (train.label==1).sum(); n = len(train)
class_weight = {0: n/(2*n0), 1: n/(2*n1)}
print('class_weight:', class_weight)
history = model.fit(train_ds, validation_data=val_ds, epochs=6, class_weight=class_weight)

### Step 7 — Evaluate honestly on the held-out TEST patients
AUC (area under ROC) is the headline number: 0.5 = guessing, 1.0 = perfect.

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, RocCurveDisplay
import matplotlib.pyplot as plt
y_true = test.label.values
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)
auc = roc_auc_score(y_true, y_prob)
acc = accuracy_score(y_true, y_pred)
print(f'TEST AUC = {auc:.3f} | accuracy = {acc:.3f}')
print('Confusion matrix [rows=true benign/malignant]:')
print(confusion_matrix(y_true, y_pred))
RocCurveDisplay.from_predictions(y_true, y_prob); plt.plot([0,1],[0,1],'--',color='grey'); plt.title('BreakHis test ROC'); plt.show()

### Step 8 — What this proves, and what it doesn't

**What was done:** trained a transfer-learning CNN to classify benign vs malignant breast
histopathology, with a leakage-safe patient-level split and class weighting.

**Why it matters:** it demonstrates the full image deep-learning pipeline (load, split, train,
evaluate) works and that histology images carry strong, learnable signal.

**Assumptions:** the chosen magnification (200x) is representative; ImageNet features transfer
to histology (they do, reasonably).

**Limitations:** this is classification, NOT survival; 82 patients is small; we only trained the
head (could fine-tune deeper). The survival question is Stage C on TCGA slides.

Copy your TEST AUC/accuracy and the ROC figure into `LAB_NOTEBOOK.md` back in the repo.